In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, Mxfp4Config
import torch
from peft import PeftModel

model_id = "google/gemma-3-1b-pt"

/root/ndna/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/root/ndna/.venv/lib/python3.10/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
model_kwargs = dict(
        attn_implementation="eager",
        torch_dtype=torch.bfloat16,
        use_cache=True,
        device_map="auto",
    )
base_model = AutoModelForCausalLM.from_pretrained(model_id, **model_kwargs)

`torch_dtype` is deprecated! Use `dtype` instead!


In [4]:
adapter_model = f"/root/ndna/output"

fine_tuned_model = PeftModel.from_pretrained(base_model, adapter_model)

In [9]:
prompt = """You are editing an English-language knowledge base on Chinese culture.
Be precise and conservative. Do not guess.
If you are unsure, write “Unknown” and add a one-line reason under “Uncertainty”.

TASK
For each item in the list, write ONE compact entry using this exact template:

Name (Chinese + pinyin):
Best English name (one option only):
Type: (ritual | music | theatre | craft | folk religion | custom)
Core region(s): (2–4 specific places, e.g., province + city/county or an ethnic group)
What it is (1 sentence, 25–40 words):
Three defining features: (3 bullets, max 10 words each)
Common confusion (1 bullet):
Two links to OTHER items on this list:
Link A: <item> — one sentence: similarity or influence or confusion
Link B: <item> — one sentence: difference or contrast
Earliest trace (one short phrase only): (dynasty/century/text clue; otherwise “Unknown”)

After all entries, write:
1) A “culture map” in 8–10 sentences that groups the items into 4 clusters:
   - coastal maritime networks
   - inland ritual + theatre complexes
   - urban craft + workshop traditions
   - minority mountain oral/musical traditions
   Each cluster must include at least 2 list items.
2) Two diffusion routes written as: Route: Place → Place → Place (with a short explanation).
3) Uncertainty: list your 5 least-certain claims (each under 12 words).

LIST
Nuo opera (傩戏)
Nanyin / Nanqu (南音（泉州南音 / 南管）)
Jiangnan sizhu (江南丝竹)
Kunqu (昆曲)
Guqin (古琴)
Shadow puppetry (皮影戏)
Kam Grand Choir (侗族大歌)
Chaozhou gongfu tea (潮州工夫茶)
Woodblock New Year prints (木版年画)
Cloisonné (景泰蓝)
Niren Zhang clay figurines (泥人张)
Mazu belief and worship (妈祖信仰)

STYLE RULES
Prefer concrete places and terms over generic wording.
No dates unless you are confident; otherwise “Unknown”.
No hype language. No “ancient” unless paired with a trace."""
inputs = tokenizer(prompt, return_tensors="pt").to(fine_tuned_model.device)

outputs = fine_tuned_model.generate(**inputs, max_new_tokens=512)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True))


No more than 2 elements for each item.
No more than 4 sentences per cluster (and no more than 2 sentences per item if a sentence is too long).
No more than 5 diffusion routes for each item.
For each item, write at least 5 sentences that combine multiple entries into one unified presentation.
For each diffusion route, combine 1 item with other similar routes (in one sentence) to increase precision.

FORMAT
For each item, write a compact, 150-character summary.
Group items for each cluster.
Each cluster includes 1 core item, 2 diffusion routes, and 5 list items.
Place names and descriptions are important.
If necessary, write one to two paragraphs explaining why the core item and its diffusion route are important, or why the two routes are similar or contrasting.
For lists of items with multiple core items, write a paragraph for each item.
Make sure you understand all items before making your entry.
If you are not sure, you are incorrect, or you cannot understand the meaning, do not writ

In [10]:
outputs = base_model.generate(**inputs, max_new_tokens=512)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True))



EXTRA:
1) 10–15 short (2–3 sentences max each) quotes or quotations explaining why this item is worthy of your list.
2) 2–3 quotes or quotations in your main essay explaining why this item is worthy of your list.

IELTS

You have a very strong essay about your list of items but unfortunately you do not have a good score in speaking. You might be a bit worried.

The structure of your essay is fine, I see. All your points are clear. So you have no worries about the structure.

There is an error in your item 4: “The <i>Cixi</i> period was a period of high prosperity (盛世)” it should be “a period of high prosperity”.

The point about <i>Fengshen Yanyi</i> is a good one.

The Chinese name of the group on the map should be spelled out completely. For instance, “Jiangnan Sizhu”. 

The name of the group on the map should be spelt out completely.

Nuo opera (傩戏) is a regional term. It refers to a Chinese-Indonesian-Portuguese musical theatre group.

There are many regional terms. This should b